# KnowMate 수집기 검증 - Step 1: 스캔 로직
## 신규 / 변경 / 삭제 파일 분류 + state.json 관리

In [ ]:
import os
import json
import uuid
from pathlib import Path
from datetime import datetime

# 테스트용 폴더 설정 (실제 경로로 변경)
WATCH_FOLDER = Path("./test_docs")
STATE_FILE   = Path("./index_state.json")

# 테스트 폴더 + 샘플 파일 생성
WATCH_FOLDER.mkdir(exist_ok=True)
(WATCH_FOLDER / "A.txt").write_text("문서 A 내용", encoding="utf-8")
(WATCH_FOLDER / "B.txt").write_text("문서 B 내용", encoding="utf-8")
print("테스트 파일 생성 완료:", list(WATCH_FOLDER.iterdir()))

## 1. state.json 읽기/쓰기

In [ ]:
def load_state(state_file: Path) -> dict:
    """저장된 스캔 상태 로드. 없으면 빈 dict 반환"""
    if not state_file.exists():
        return {}
    with open(state_file, "r", encoding="utf-8") as f:
        return json.load(f)

def save_state(state_file: Path, state: dict) -> None:
    """현재 스캔 상태를 state.json에 저장"""
    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
    print(f"state.json 저장 완료: {len(state)}건")

# 초기 상태 확인
state = load_state(STATE_FILE)
print("현재 state:", state)

## 2. 현재 디스크 파일 수집 + mtime/size 추출

In [ ]:
SUPPORTED_EXT = {".docx", ".doc", ".xlsx", ".xls", ".pptx", ".ppt", ".pdf", ".txt"}

def scan_folder(folder: Path) -> dict:
    """폴더 내 지원 확장자 파일의 mtime+size 수집"""
    current = {}
    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXT:
            stat = path.stat()
            current[str(path)] = {
                "mtime": stat.st_mtime,
                "size" : stat.st_size,
            }
    return current

current_files = scan_folder(WATCH_FOLDER)
print("디스크 현재 파일:")
for path, info in current_files.items():
    print(f"  {path} | mtime={info['mtime']:.0f} | size={info['size']}bytes")

## 3. 신규 / 변경 / 삭제 분류

In [ ]:
def classify_changes(saved: dict, current: dict) -> tuple[list, list, list]:
    """
    saved:   이전 state.json 내용
    current: 현재 디스크 스캔 결과
    반환: (신규, 변경, 삭제) 파일 경로 리스트
    """
    saved_paths   = set(saved.keys())
    current_paths = set(current.keys())

    new_files     = list(current_paths - saved_paths)
    deleted_files = list(saved_paths - current_paths)
    modified_files = [
        p for p in current_paths & saved_paths
        if current[p]["mtime"] != saved[p]["mtime"]
        or current[p]["size"]  != saved[p]["size"]
    ]
    return new_files, modified_files, deleted_files

new_files, modified_files, deleted_files = classify_changes(state, current_files)
print(f"신규:  {new_files}")
print(f"변경:  {modified_files}")
print(f"삭제:  {deleted_files}")

## 4. 최초 스캔 — state.json 초기화

In [ ]:
def build_initial_state(current: dict) -> dict:
    """최초 스캔: 모든 파일을 신규로 등록, chunk_ids는 빈 리스트"""
    state = {}
    for path, info in current.items():
        state[path] = {
            "mtime"     : info["mtime"],
            "size"      : info["size"],
            "indexed_at": datetime.now().isoformat(),
            "chunk_ids" : []  # 임베딩 후 채워짐
        }
    return state

initial_state = build_initial_state(current_files)
save_state(STATE_FILE, initial_state)
print("\n저장된 state 내용:")
print(json.dumps(initial_state, ensure_ascii=False, indent=2))

## 5. 증분 스캔 시뮬레이션 — 파일 추가/수정/삭제

In [ ]:
import time

# 시뮬레이션: C.txt 신규 추가
(WATCH_FOLDER / "C.txt").write_text("문서 C 신규", encoding="utf-8")

# 시뮬레이션: A.txt 내용 수정 (mtime 변경)
time.sleep(0.1)
(WATCH_FOLDER / "A.txt").write_text("문서 A 수정된 내용", encoding="utf-8")

# 시뮬레이션: B.txt 삭제
(WATCH_FOLDER / "B.txt").unlink()

# 재스캔
saved_state   = load_state(STATE_FILE)
current_files = scan_folder(WATCH_FOLDER)
new_files, modified_files, deleted_files = classify_changes(saved_state, current_files)

print(f"신규:  {[Path(p).name for p in new_files]}")
print(f"변경:  {[Path(p).name for p in modified_files]}")
print(f"삭제:  {[Path(p).name for p in deleted_files]}")
print("\n✅ 기대값: 신규=[C.txt] 변경=[A.txt] 삭제=[B.txt]")

## 6. chunk_ids 추적 확인
삭제/변경된 파일의 chunk_ids를 꺼내서 벡터DB에서 제거할 대상 확인

In [ ]:
# 실제로는 임베딩 후 chunk_ids가 채워지지만
# 여기서는 mock chunk_ids로 추적 로직만 검증
mock_state = load_state(STATE_FILE)

# A.txt에 mock chunk_ids 주입
a_path = str(WATCH_FOLDER / "A.txt")
b_path = str(WATCH_FOLDER / "B.txt")
if a_path in mock_state:
    mock_state[a_path]["chunk_ids"] = [str(uuid.uuid4()), str(uuid.uuid4())]
if b_path in mock_state:
    mock_state[b_path]["chunk_ids"] = [str(uuid.uuid4())]

# 변경/삭제 파일의 chunk_ids 추출
chunks_to_delete = []
for path in modified_files + deleted_files:
    if path in mock_state:
        chunks_to_delete.extend(mock_state[path]["chunk_ids"])

print("벡터DB에서 제거할 chunk_ids:")
for cid in chunks_to_delete:
    print(f"  {cid}")
print("\n✅ 이 ID들을 LanceDB에서 삭제하면 됨")

## 정리
- `scan_folder()` : 디스크 현재 상태 수집
- `classify_changes()` : 신규/변경/삭제 분류
- `build_initial_state()` : 최초 state.json 생성
- `chunk_ids` : 변경/삭제 시 벡터DB 정리 대상 추적

**다음 Step 2**: orphan 안전장치 (폴더 루트 가드 + 30% 대량삭제 차단 + soft delete)